# Key RL Equations: *A Single Goal is All You Need*

> **Paper**: Liu, G., Tang, M., & Eysenbach, B. (2024). *A Single Goal is All You Need: Skills and Exploration Emerge from Contrastive RL without Rewards, Demonstrations, or Subgoals.* arXiv:2408.05804

This notebook walks through every important equation in the paper, explains the variables involved, and builds up an intuitive understanding of how the algorithm works — no prior familiarity with contrastive RL is assumed.

---
## Overview

The paper proposes a surprisingly simple idea: **give the agent only one goal observation and always command it to reach that goal during training**. No reward function, no demonstrations, no hand-crafted subgoal curricula. Remarkably, complex skills and directed exploration *emerge automatically*.

The algorithm is built on **Contrastive RL (CRL)**, which uses a contrastive (InfoNCE) loss to learn a goal-conditioned Q-function. The single modification is that *data collection always uses the single hard goal* instead of a curriculum of easier goals.

We will cover five components (spanning all four numbered equations in the paper) in order:

| # | Component | Equations / Algorithm |
|---|-----------|----------------------|
| 1 | MDP Setup & Objective | Eq. (1) state occupancy measure; Eq. (2) training objective |
| 2 | Goal Conditioning | Inner-product critic $C(s,a,s_f) = \\phi(s,a)^\\top\\psi(s_f)$ |
| 3 | Exploration Strategy | Algorithm 1, line 3 (always command $s^*$) |
| 4 | Critic Update | Eq. (3) InfoNCE + LogSumExp regularisation |
| 5 | Policy (Actor) Update | Eq. (4) MaxEnt goal-conditioned actor loss |

---

## 1  MDP Setup and Objective

### 1.1  Controlled Markov Process

The environment is modelled as a **controlled Markov process** — an MDP *without* a reward function:

| Symbol | Meaning |
|--------|---------|
| $s_t$ | State of the environment at time step $t$ |
| $a_t$ | Action taken by the agent at time step $t$ |
| $p_0(s_0)$ | Distribution from which the **initial state** $s_0$ is sampled |
| $p(s_{t+1} \mid s_t, a_t)$ | **Transition dynamics**: probability of the next state given the current state and action |
| $s^*$ | The single **target goal state** provided to the agent |
| $\pi(a_t \mid s_t)$ | The agent's **policy**: a mapping from states to actions |
| $\gamma \in [0,1)$ | **Discount factor**: weights future states less than near-term states |

The dynamics are **Markovian** — the next state depends *only* on the current state and action, not on the full history.

---

### 1.2  γ-Discounted State Occupancy Measure — Equation (1)

$$
\rho^\pi(s_f) \;\triangleq\; (1 - \gamma)\sum_{t=0}^{\infty} \gamma^t \, p^\pi_t(s_t = s_f)
\tag{1}
$$

**What it means:**  
$\rho^\pi(s_f)$ measures *how much time* the policy $\pi$ spends at state $s_f$, weighting near-future visits more heavily than distant ones (via $\gamma^t$).

| Symbol | Meaning |
|--------|---------|
| $\rho^\pi(s_f)$ | Discounted probability (density) of visiting state $s_f$ under policy $\pi$ |
| $(1 - \gamma)$ | Normalisation constant that ensures $\rho^\pi$ integrates (sums) to 1 |
| $\gamma^t$ | Exponential discount: visits at time $t$ are worth $\gamma^t$ times a visit at $t=0$ |
| $p^\pi_t(s_t = s_f)$ | Probability of being in state $s_f$ at time $t$ when following policy $\pi$ |

**Intuition:** Think of $\rho^\pi(s_f)$ as a weighted histogram of all the states the agent visits. States visited early get more weight than states visited late (because $\gamma < 1$). If $\gamma = 0.99$, a state visited at $t=100$ is worth only $0.99^{100} \approx 0.37$ of a state visited at $t=0$.

---

### 1.3  Training Objective — Equation (2)

$$
\max_{\pi} \; \rho^\pi(s_f = s^*)
\tag{2}
$$

**What it means:**  
Find the policy $\pi$ that *maximises* the time spent at the target goal $s^*$.

| Symbol | Meaning |
|--------|---------|
| $s^*$ | The single fixed **goal state** provided to the algorithm (the only human input) |
| $\rho^\pi(s_f = s^*)$ | The discounted fraction of time spent at $s^*$ under policy $\pi$ |

**Relation to standard RL:**  
- In **discrete** state spaces, this is equivalent to maximising expected discounted return with reward $r(s_t, a_t) = \mathbf{1}[s_t = s^*]$ (a sparse +1 reward only at the goal).  
- In **continuous** state spaces, it is equivalent to a reward $r(s_t, a_t) = p(s' = s^* \mid s_t, a_t)$ — the probability of transitioning *exactly* to the goal.

**Key point:** The objective maximises the *time at the goal*, not just whether the goal is ever reached. This naturally encourages the agent to stay near the goal once it finds it.

---

## 2  Goal Conditioning

The paper uses a **goal-conditioned policy** and a **goal-conditioned value function**. Instead of learning a single policy, the agent learns a family of policies parameterised by a goal:

$$
\pi(a_t \mid s_t, g)
$$

| Symbol | Meaning |
|--------|---------|
| $g$ | A **goal state** the agent is trying to reach |
| $\pi(a_t \mid s_t, g)$ | The probability of taking action $a_t$ when in state $s_t$ and aiming for goal $g$ |

**Why multi-goal training for a single-goal problem?**  
Although the agent is ultimately evaluated on just one goal $s^*$, the critic is trained with *many* randomly sampled goals from the replay buffer (future states as goals). This is crucial: ablation experiments in the paper show that training the actor with only $s^*$ *degrades* performance. Learning with many goals gives the critic a richer training signal and prevents overfitting to a single goal state.

The goal-conditioned **critic** is written as:

$$
C(s, a, s_f) \;=\; \phi(s, a)^\top \psi(s_f)
$$

| Symbol | Meaning |
|--------|---------|
| $C(s, a, s_f)$ | Critic output: (log) likelihood that the agent visits state $s_f$ starting from $(s, a)$ |
| $\phi(s, a)$ | Learned **state-action embedding** (a neural network encoder) |
| $\psi(s_f)$ | Learned **goal embedding** (a separate neural network encoder) |
| $\phi(s, a)^\top \psi(s_f)$ | Dot product between the two embeddings — the critic score |

**Interpretation of the critic:**  
The dot product critic encodes a Q-value: $\phi(s,a)^\top \psi(s_f) = \log Q(s, a, s_f) - \log \rho(s_f)$.  
High dot product → the policy is *likely* to visit $s_f$ when starting from $(s, a)$.  
Low (negative) dot product → unlikely to visit $s_f$.

---

## 3  Exploration Strategy

### The Key Idea (Algorithm 1, Line 3)

```
Algorithm 1 — Single-goal Exploration with Contrastive RL
─────────────────────────────────────────────────────────
Input : single target goal s*
1: Initialise policy π_θ(a | s, g), replay buffer B,
   critic logits φ(s, a)ᵀ ψ(sᶠ)
2: while not converged do
3:   Collect one trajectory using π(a | s, sᶠ = s*)  ← always use the hard goal
         and add it to buffer B
4:   Update φ(s,a), ψ(sᶠ) and π using Contrastive RL
      (Eq. 3 for the critic, Eq. 4 for the actor)
5: Return policy π(a | s, g = s*)
```

**The single change from standard CRL:**  
Most prior methods sample goals for exploration from a *curriculum* (easy → hard). This algorithm always commands the **single hard goal $s^*$** during data collection (line 3). The critic and actor losses are otherwise identical to CRL.

**Why does this drive exploration?**  
- Early in training the policy has never seen $s^*$, so the critic gives *incorrect* (but non-trivial) values.  
- The inner-product critic $\phi(s,a)^\top \psi(s^*)$ steers the agent toward states whose embeddings *align* with the goal embedding, even before the goal is ever reached.  
- This alignment signal changes continuously as the representations are updated, creating a dynamic exploration incentive — without any explicit exploration bonus.  
- Once the goal is reached consistently, the critic becomes accurate, and the policy stabilises into an exploitation strategy.

---

## 4  Critic Update — InfoNCE Contrastive Loss — Equation (3)

The critic embeddings $\phi$ and $\psi$ are trained with a **contrastive learning objective**:

$$
\max_{\phi,\, \psi} \;
\mathbb{E}_{\substack{(s,a) \sim p(s,a) \\ s_f^{(1)} \sim \rho(s_f \mid s, a) \\ s_f^{(2:N)} \sim \rho(s_f)}}
\left[
  \underbrace{\log \frac{e^{\phi(s,a)^\top \psi(s_f^{(1)})}}
                         {\sum_{j=1}^{N} e^{\phi(s,a)^\top \psi(s_f^{(j)})}}}_{\text{InfoNCE}}
  \;-\;
  0.01 \cdot
  \underbrace{\log\!\left(\sum_{j=1}^{N} e^{\phi(s,a)^\top \psi(s_f^{(j)})}\right)^{\!2}}_{\text{LogSumExp regularisation}}
\right]
\tag{3}
$$

### Variable Guide

| Symbol | Meaning |
|--------|---------|
| $p(s, a)$ | Marginal distribution of state-action pairs in the **replay buffer** |
| $\rho(s_f \mid s, a)$ | Empirical discounted state occupancy measure conditioned on $(s,a)$ — i.e., the distribution of future states the agent actually visits after $(s,a)$ |
| $s_f^{(1)}$ | A **positive example**: a future state genuinely reached from $(s, a)$, sampled by looking $\Delta \sim \text{Geom}(1-\gamma)$ steps ahead in the trajectory |
| $s_f^{(2:N)}$ | **Negative examples**: future states sampled from the *marginal* distribution $\rho(s_f)$, obtained by shuffling future states across different trajectories in the replay buffer |
| $N$ | Batch size (total number of future-state candidates = 1 positive + $N-1$ negatives) |
| $\phi(s,a)^\top \psi(s_f^{(j)})$ | Dot product (critic score) for the $j$-th candidate future state |

### Breaking Down the Two Terms

#### Term 1 — InfoNCE Loss

$$
\log \frac{e^{\phi(s,a)^\top \psi(s_f^{(1)})}}{\sum_{j=1}^{N} e^{\phi(s,a)^\top \psi(s_f^{(j)})}}
$$

This is a **contrastive classification loss**: the model should assign the *highest score* to the true future state $s_f^{(1)}$ compared to the $N-1$ negative examples. Maximising this log-probability trains the embeddings so that:
- States that are *genuinely* reachable from $(s, a)$ get **high dot products**.
- States that are *not* reachable get **low dot products**.

This is conceptually identical to contrastive losses used in computer vision (e.g., SimCLR, MoCo), but the "positive pair" here is a *(current state, future state)* pair from the *same* trajectory.

#### Term 2 — LogSumExp Regularisation

$$
-\, 0.01 \cdot \log\!\left(\sum_{j=1}^{N} e^{\phi(s,a)^\top \psi(s_f^{(j)})}\right)^{\!2}
$$

This term **prevents the dot products from growing unboundedly**. Without it, the model can trivially maximise InfoNCE by making all dot products very large (since the softmax denominator cancels with the numerator). The small coefficient $0.01$ keeps the regularisation mild. Prior theoretical analysis of CRL shows this term is necessary for the learned Q-values to be well-calibrated.

### Practical Sampling

In practice, for each training step:
1. Sample a transition $(s_t, a_t)$ from the replay buffer.
2. Sample a future time offset $\Delta \sim \text{Geometric}(1 - \gamma)$ and set $s_f^{(1)} = s_{t+\Delta}$ (the positive).
3. Obtain negatives $s_f^{(2:N)}$ by taking future states from *other* samples in the same batch (shuffling).
4. Compute the loss in Eq. (3) and update $\phi$, $\psi$.

The geometric distribution over $\Delta$ ensures that the sampled future states match the $\gamma$-discounted occupancy measure.

---

## 5  Policy (Actor) Update — Equation (4)

Once the critic embeddings $\phi$ and $\psi$ are learned, the policy is updated by maximising the (log) Q-value:

$$
\max_{\pi} \;
\mathbb{E}_{\substack{p(s)\, p(g) \\ \pi(a \mid s, g)}}
\Big[
  \phi(s, a)^\top \psi(g)
  \;+\;
  \alpha\, \mathcal{H}\bigl(\pi(\cdot \mid s, g)\bigr)
\Big]
\tag{4}
$$

### Variable Guide

| Symbol | Meaning |
|--------|---------|
| $p(s)$ | Distribution of states sampled from the **replay buffer** |
| $p(g)$ | Distribution of **goal states** sampled from the replay buffer (future states act as training goals) |
| $\pi(a \mid s, g)$ | The **goal-conditioned policy** (actor) being optimised |
| $\phi(s, a)^\top \psi(g)$ | The **critic score**: alignment between the state-action embedding and the goal embedding; a proxy for the log Q-value |
| $\mathcal{H}(\pi(\cdot \mid s, g))$ | **Policy entropy**: $-\sum_a \pi(a \mid s, g) \log \pi(a \mid s, g)$ — a measure of how random the policy is |
| $\alpha$ | **Entropy temperature** (adaptive coefficient): balances exploration (high entropy) vs. exploitation (low entropy); tuned automatically using SAC's entropy tuning |

### Breaking Down the Two Terms

#### Term 1 — Q-value Maximisation

$$
\phi(s, a)^\top \psi(g)
$$

The policy should select actions $a$ that maximise the dot product between the state-action embedding and the goal embedding. High dot product means the action is likely to lead the agent to goal $g$. This is equivalent to maximising the goal-conditioned Q-value: $\phi(s,a)^\top \psi(g) \approx \log Q(s,a,g)$.

#### Term 2 — Entropy Regularisation (MaxEnt RL)

$$
\alpha\, \mathcal{H}\bigl(\pi(\cdot \mid s, g)\bigr)
$$

Encouraging **high entropy** prevents the policy from collapsing to a single deterministic action too early. This is the **Soft Actor-Critic (SAC)** entropy bonus, which:
- Promotes **exploration** — a stochastic policy tries diverse actions.
- Improves **robustness** — the policy doesn't over-commit to one path.
- Is automatically calibrated via the adaptive $\alpha$ so the policy entropy stays near a target value.

### Multi-Goal Actor Training (Important Detail)

Even though the agent **always collects data by commanding goal $s^*$** (exploration), the actor loss trains the policy on **many goals** sampled from the replay buffer ($p(g)$). This multi-task training is crucial: it prevents the critic from only learning about paths to $s^*$ and gives it a richer understanding of the entire state space — which in turn provides a better learning signal for exploration.

---

## 6  Full Algorithm Summary

Putting everything together:

```
──────────────────────────────────────────────────────────────────────────────
 Algorithm 1: Single-Goal Exploration with Contrastive RL
──────────────────────────────────────────────────────────────────────────────
 Input:  Single target goal state s*
 Init:   Policy π_θ(a | s, g)            ← goal-conditioned actor (SAC-style)
         Critic embeddings φ(s, a), ψ(sᶠ) ← separate encoder networks
         Replay buffer B                  ← stores (s, a, s_next) tuples

 while not converged do:

   ┌── EXPLORATION (data collection) ──────────────────────────────────────┐
   │  Run one episode using π(a | s_t, g = s*)  ← ALWAYS the hard goal    │
   │  Record trajectory τ = {(s₀,a₀), (s₁,a₁), …}                         │
   │  Add τ to replay buffer B                                              │
   └────────────────────────────────────────────────────────────────────────┘

   ┌── CRITIC UPDATE (Eq. 3) ───────────────────────────────────────────────┐
   │  Sample batch {(s, a)} from B                                          │
   │  For each (s, a):                                                      │
   │    Positive: sᶠ⁺ = s_{t+Δ}, Δ ~ Geom(1-γ)  (true future state)       │
   │    Negatives: sᶠ⁻¹…ᴺ⁻¹ shuffled from the batch  (random future states)│
   │  Maximise InfoNCE + LogSumExp loss (Eq. 3)                             │
   │  → φ and ψ now encode: "how likely is sᶠ reachable from (s,a)?"       │
   └────────────────────────────────────────────────────────────────────────┘

   ┌── ACTOR UPDATE (Eq. 4) ────────────────────────────────────────────────┐
   │  Sample states s ~ B, goals g ~ B (random future states as goals)      │
   │  For each (s, g):                                                      │
   │    Sample action a ~ π(· | s, g)                                       │
   │    Maximise: φ(s, a)ᵀ ψ(g) + α H(π(·|s,g))                           │
   │  → π learns to reach many goals, not just s*                           │
   └────────────────────────────────────────────────────────────────────────┘

 Return: π(a | s, g = s*)  ← deploy the policy commanded to reach s*
──────────────────────────────────────────────────────────────────────────────
```

### What makes this algorithm work?

| Property | Explanation |
|----------|-------------|
| **Self-directed exploration** | Commanding the hard goal creates misaligned embeddings early in training, pushing the policy to explore new states until the embeddings align. Once the goal is reached reliably, the Q-values become accurate and the policy stabilises. |
| **No extra hyperparameters** | Unlike ε-greedy or Gaussian noise exploration (which require a noise scale and decay schedule), the exploration emerges entirely from the contrastive critic. |
| **Curriculum-free** | Prior methods need a hand-designed curriculum from easy to hard goals. This algorithm gets equivalent or better performance with just one goal. |
| **Multi-task critic, single-goal data** | Training the critic/actor on many replay goals (Eq. 4) provides rich supervision, while data collection uses only $s^*$ (Algorithm 1, line 3). This separation is key. |
| **Inner-product critic is crucial** | Ablations show that replacing $\phi(s,a)^\top\psi(s_f)$ with a monolithic $Q(s,a,s_f)$ network eliminates the emergent exploration — the factored representation structure drives it. |

---

## 7  Equation Reference Card

A compact summary of all key equations (Eq. 1–4) and the exploration rule from Algorithm 1:

---

**Eq. (1) — State occupancy measure**
$$
\rho^\pi(s_f) = (1-\gamma) \sum_{t=0}^{\infty} \gamma^t \, p^\pi_t(s_t = s_f)
$$
*Discounted probability of visiting $s_f$ under policy $\pi$.*

---

**Eq. (2) — Training objective**
$$
\max_{\pi} \; \rho^\pi(s_f = s^*)
$$
*Find the policy that spends the most time at the target goal $s^*$.*

---

**Eq. (3) — Critic loss (InfoNCE + LogSumExp)**
$$
\max_{\phi,\psi}\; \mathbb{E}\!\left[
  \log \frac{e^{\phi(s,a)^\top \psi(s_f^{(1)})}}
             {\sum_{j=1}^{N} e^{\phi(s,a)^\top \psi(s_f^{(j)})}}
  - 0.01 \cdot \log\!\left(\sum_{j=1}^N e^{\phi(s,a)^\top \psi(s_f^{(j)})}\right)^{\!2}
\right]
$$
*Trains embeddings so that reachable future states get high dot-product scores.*

---

**Eq. (4) — Actor loss (MaxEnt goal-conditioned RL)**
$$
\max_{\pi}\; \mathbb{E}_{p(s)\,p(g)\,\pi(a|s,g)}\!\Big[
  \phi(s,a)^\top \psi(g) + \alpha\,\mathcal{H}(\pi(\cdot|s,g))
\Big]
$$
*Trains the policy to select actions that reach sampled goals, while staying stochastic.*

---

**Algorithm 1 — Exploration rule**
$$
\text{Collect trajectory: } a_t \sim \pi(\cdot \mid s_t,\; g = s^*) \quad \forall t
$$
*Always command the single hard goal $s^*$ during data collection.*

---